# Skin Disease Classification using EfficientNetB0

EfficientNetB0 is specifically designed for image-based tasks and is highly suitable for facial skin condition analysis. Despite being a deep convolutional neural network, it remains computationally efficient due to its optimized scaling approach. Compared to traditional models, it achieves higher accuracy with fewer parameters, resulting in faster and more efficient training. Therefore, it provides a balance between performance and speed, making it appropriate for this study.

---
## Upload ZIP Files

Upload all zip files at once (training, testing, and unseen).
The cell below will automatically extract and reorganize them into:
```
final_dataset/
  train/
    acne/
    eczema/
    ...
  test/
    acne/
    ...
  unseen/
    acne/
    ...
```

In [ ]:
!pip install -q pillow

import os, zipfile, shutil
from PIL import Image, UnidentifiedImageError
from google.colab import files

RAW_DIR    = '/content/raw_zips'
FINAL_DIR  = '/content/final_dataset'
TRAIN_DIR  = os.path.join(FINAL_DIR, 'train')
TEST_DIR   = os.path.join(FINAL_DIR, 'test')
UNSEEN_DIR = os.path.join(FINAL_DIR, 'unseen')

for d in [RAW_DIR, TRAIN_DIR, TEST_DIR, UNSEEN_DIR]:
    os.makedirs(d, exist_ok=True)

# Accepts all common image formats
VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.tif', '.webp'}

print('Select all zip files (training, testing, and unseen):')
uploaded = files.upload()

for fname in uploaded:
    dest = os.path.join(RAW_DIR, fname)
    if not os.path.exists(dest):
        shutil.move(fname, dest)
    print(f'  {fname}')

print(f'\n{len(uploaded)} zip(s) uploaded.')

Select all zip files (training, testing, and unseen):


TypeError: 'NoneType' object is not subscriptable

---
## Extract and Restructure

Extracts images from every zip into the correct train, test, or unseen folder.
- Accepts all image types (jpg, png, bmp, gif, tiff, webp)
- Skips non-image and corrupt files silently
- Converts all images to RGB JPEG for consistency

In [ ]:
ZIP_MAP = {
    # Batch 1
    'acne-training.zip':                        ('train',  'acne'),
    'acne-testing.zip':                         ('test',   'acne'),
    'acne-unseen.zip':                          ('unseen', 'acne'),
    'benign-kerastosis-training.zip':           ('train',  'benign_kerastosis'),
    'benign-kerastosis-testing.zip':            ('test',   'benign_kerastosis'),
    'benign-kerastosis-unseen.zip':             ('unseen', 'benign_kerastosis'),
    'actinic-kerastosis-training.zip':          ('train',  'actinic_kerastosis'),
    'actinic-kerastosis-testing.zip':           ('test',   'actinic_kerastosis'),
    'actinic-kerastosis-unseen.zip':            ('unseen', 'actinic_kerastosis'),
    'rosacea-training.zip':                     ('train',  'rosacea'),
    'rosacea-testing.zip':                      ('test',   'rosacea'),
    'rosacea-unseen.zip':                       ('unseen', 'rosacea'),
    'atopic-dermatitis-training.zip':           ('train',  'atopic_dermatitis'),
    'atopic-dermatitis-testing.zip':            ('test',   'atopic_dermatitis'),
    'atopic-dermatitis-unseen.zip':             ('unseen', 'atopic_dermatitis'),
    'bullous-disease-training.zip':             ('train',  'bullous_disease'),
    'bullous-disease-testing.zip':              ('test',   'bullous_disease'),
    'bullous-disease-unseen.zip':               ('unseen', 'bullous_disease'),
    'cellulitis-impetigo-training.zip':         ('train',  'cellulitis'),
    'cellulitis-impetigo-testing.zip':          ('test',   'cellulitis'),
    'cellulitis-impetigo-unseen.zip':           ('unseen', 'cellulitis'),
    'basal-cell-carcinoma-training.zip':        ('train',  'basal_cell_carcinoma'),
    'basal-cell-carcinoma-testing.zip':         ('test',   'basal_cell_carcinoma'),
    'basal-cell-carcinoma-unseen.zip':          ('unseen', 'basal_cell_carcinoma'),
    'melanocytic-nevi-training.zip':            ('train',  'melanocytic_nevi'),
    'melanocytic-nevi-testing.zip':             ('test',   'melanocytic_nevi'),
    'melanocytic-nevi-unseen.zip':              ('unseen', 'melanocytic_nevi'),
    # Batch 2
    'Lupus-and-other-connective-tissue-training.zip':            ('train',  'lupus'),
    'Lupus-and-other-connective-tissue-testing.zip':             ('test',   'lupus'),
    'Lupus-and-other-connective-tissue-unseen.zip':              ('unseen', 'lupus'),
    'melanoma-skin-cancer-and-moles-training.zip':               ('train',  'melanoma'),
    'melanoma-skin-cancer-and-moles-testing.zip':                ('test',   'melanoma'),
    'melanoma-skin-cancer-and-moles-unseen.zip':                 ('unseen', 'melanoma'),
    'poison-ivy-training.zip':                  ('train',  'poison_ivy'),
    'poison-ivy-testing.zip':                   ('test',   'poison_ivy'),
    'poison-ivy-unseen.zip':                    ('unseen', 'poison_ivy'),
    'psoriasis-pictures-training.zip':          ('train',  'psoriasis'),
    'psoriasis-pictures-testing.zip':           ('test',   'psoriasis'),
    'psoriasis-pictures-unseen.zip':            ('unseen', 'psoriasis'),
    'urticaria-hives-training.zip':             ('train',  'urticaria_hives'),
    'urticaria-hives-testing.zip':              ('test',   'urticaria_hives'),
    'urticaria-hives-unseen.zip':               ('unseen', 'urticaria_hives'),
    'vascular-tumors-training.zip':             ('train',  'vascular_tumors'),
    'vascular-tumors-testing.zip':              ('test',   'vascular_tumors'),
    'vascular-tumors-unseen.zip':               ('unseen', 'vascular_tumors'),
    'vasculitis-photo-training.zip':            ('train',  'vasculitis'),
    'vasculitis-photo-testing.zip':             ('test',   'vasculitis'),
    'vasculitis-photo-unseen.zip':              ('unseen', 'vasculitis'),
    'wart-molluscum-and-other-viral-infection-training.zip': ('train',  'warts_molluscum'),
    'wart-molluscum-and-other-viral-infection-testing.zip':  ('test',   'warts_molluscum'),
    'wart-molluscum-and-other-viral-infection-unseen.zip':   ('unseen', 'warts_molluscum'),
    # Batch 3
    'Eczema-Training.zip':                      ('train',  'eczema'),
    'Eczema-Testing.zip':                       ('test',   'eczema'),
    'Eczema-Unseen.zip':                        ('unseen', 'eczema'),
    'Exanthems and Drug Eruptions-Training.zip': ('train',  'exanthems'),
    'Exanthems and Drug Eruptions-Testing.zip':  ('test',   'exanthems'),
    'Exanthems and Drug Eruptions-Unseen.zip':   ('unseen', 'exanthems'),
    'Herpes HPV and other STDs Photos-Training.zip': ('train',  'hpv'),
    'Herpes HPV and other STDs Photos-Testing.zip':  ('test',   'hpv'),
    'Herpes HPV and other STDs Photos-Unseen.zip':   ('unseen', 'hpv'),
    'Light Diseases and Disorders of Pigmentation-Training.zip': ('train',  'light_disease'),
    'Light Diseases and Disorders of Pigmentation-Testing.zip':  ('test',   'light_disease'),
    'Light Diseases and Disorders of Pigmentation-Unseen.zip':   ('unseen', 'light_disease'),
    'Scabies Lyme Disease and other Infestations and Bites-Training.zip': ('train',  'scabies'),
    'Scabies Lyme Disease and other Infestations and Bites-Testing.zip':  ('test',   'scabies'),
    'Scabies Lyme Disease and other Infestations and Bites-Unseen.zip':   ('unseen', 'scabies'),
    'Seborrheic Keratoses and other Benign Tumors-Training.zip': ('train',  'seborrheic_kerastoses'),
    'Seborrheic Keratoses and other Benign Tumors-Testing.zip':  ('test',   'seborrheic_kerastoses'),
    'Seborrheic Keratoses and other Benign Tumors-Unseen.zip':   ('unseen', 'seborrheic_kerastoses'),
    'Systemic Disease-Training.zip':            ('train',  'systemic_disease'),
    'Systemic Disease-Testing.zip':             ('test',   'systemic_disease'),
    'Systemic Disease-Unseen.zip':              ('unseen', 'systemic_disease'),
    'Tinea Ringworm Candidiasis and other Fungal Infections-Training.zip': ('train',  'tinea_ringworm'),
    'Tinea Ringworm Candidiasis and other Fungal Infections-Testing.zip':  ('test',   'tinea_ringworm'),
    'Tinea Ringworm Candidiasis and other Fungal Infections-Unseen.zip':   ('unseen', 'tinea_ringworm'),
}


def safe_extract(zip_path, split, label):
    dest = os.path.join(FINAL_DIR, split, label)
    os.makedirs(dest, exist_ok=True)
    extracted = 0
    skipped   = 0
    seen      = set()
    with zipfile.ZipFile(zip_path, 'r') as z:
        for member in z.namelist():
            if member.endswith('/') or os.path.basename(member).startswith('.'):
                continue
            if os.path.splitext(member)[1].lower() not in VALID_EXT:
                skipped += 1
                continue
            base = os.path.splitext(os.path.basename(member))[0]
            out_name = base + '.jpg'
            c = 1
            while out_name in seen:
                out_name = f'{base}_{c}.jpg'
                c += 1
            seen.add(out_name)
            try:
                with z.open(member) as f:
                    img = Image.open(f)
                    img.verify()
                with z.open(member) as f:
                    img = Image.open(f).convert('RGB')
                    img.save(os.path.join(dest, out_name), 'JPEG', quality=95)
                extracted += 1
            except Exception:
                skipped += 1
    return extracted, skipped


total_ok, total_skip = 0, 0
not_found = []

for zip_name, (split, label) in ZIP_MAP.items():
    zip_path = os.path.join(RAW_DIR, zip_name)
    if not os.path.exists(zip_path):
        not_found.append(zip_name)
        continue
    n_ok, n_skip = safe_extract(zip_path, split, label)
    total_ok   += n_ok
    total_skip += n_skip
    print(f'[{split:6s}] {label:<30} {n_ok} images  ({n_skip} skipped)')

print(f'\nTotal extracted : {total_ok}')
print(f'Total skipped   : {total_skip}  (non-image or corrupt files)')
if not_found:
    print(f'\nNot uploaded ({len(not_found)}):')
    for z in not_found:
        print(f'  - {z}')

---
## Verify Structure

Displays image counts per class per split to confirm the dataset was restructured correctly.

In [ ]:
all_classes = sorted(os.listdir(TRAIN_DIR)) if os.path.exists(TRAIN_DIR) else []

print(f"{'CLASS':<30} {'TRAIN':>7} {'TEST':>7} {'UNSEEN':>7}")
print('=' * 55)
t_tot = te_tot = u_tot = 0
for cls in all_classes:
    t  = len(os.listdir(os.path.join(TRAIN_DIR,  cls))) if os.path.isdir(os.path.join(TRAIN_DIR,  cls)) else 0
    te = len(os.listdir(os.path.join(TEST_DIR,   cls))) if os.path.isdir(os.path.join(TEST_DIR,   cls)) else 0
    u  = len(os.listdir(os.path.join(UNSEEN_DIR, cls))) if os.path.isdir(os.path.join(UNSEEN_DIR, cls)) else 0
    t_tot += t; te_tot += te; u_tot += u
    print(f'  {cls:<28} {t:>7} {te:>7} {u:>7}')
print('=' * 55)
print(f"  {'TOTAL':<28} {t_tot:>7} {te_tot:>7} {u_tot:>7}")
print(f'\n{len(all_classes)} classes found')

---
## Import Libraries

These libraries are required to build the CNN architecture, connect the layers, configure the optimizer, and train the model.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, BatchNormalization, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report

---
## Dataset Parameters

Sets the image input size and detects the number of classes from the training folder automatically.

In [ ]:
IMG_SIZE = 224
# Image size (224x224 pixels)
# Required input size for EfficientNetB0

BATCH_SIZE = 32

NUM_CLASSES = len([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
# Total number of skin disease categories
# Auto-detected from the number of class folders in the training directory

print(f'IMG_SIZE    = {IMG_SIZE}')
print(f'BATCH_SIZE  = {BATCH_SIZE}')
print(f'NUM_CLASSES = {NUM_CLASSES}')

---
## Data Generators

Prepares image batches for training, testing, and unseen evaluation.

EfficientNetB0 has built-in preprocessing through preprocess_input. Using rescale=1./255 instead would conflict with EfficientNet's internal normalization and reduce accuracy.

In [ ]:
# Training generator with augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    # EfficientNet's own normalization
    rotation_range=20,
    # Randomly rotate images up to 20 degrees
    zoom_range=0.2,
    # Randomly zoom in or out
    width_shift_range=0.1,
    height_shift_range=0.1,
    # Randomly shift images horizontally and vertically
    horizontal_flip=True
    # Randomly flip images left to right
)

# Testing and unseen generators with no augmentation
eval_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

test_data = eval_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

unseen_data = eval_datagen.flow_from_directory(
    UNSEEN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f'train  : {train_data.samples} images, {train_data.num_classes} classes')
print(f'test   : {test_data.samples} images')
print(f'unseen : {unseen_data.samples} images')

---
## Designing Layers

This section defines the architecture of the model.

EfficientNetB0 serves as the base feature extractor. It captures important image patterns such as edges, textures, and shapes without being too computationally heavy. Custom classification layers are added on top, using ReLU activations for learning deeper features and Softmax at the output layer for multi-class prediction.

The depth of the model comes from combining a pretrained base model, a pooling layer, batch normalization, dense layers, dropout regularization, and a final output layer. This structure allows the model to learn complex patterns while remaining efficient and stable during training.

In [ ]:
# Base Model (EfficientNetB0)

base_model = EfficientNetB0(
    weights='imagenet',
    # Load pretrained weights from ImageNet
    # This allows the model to use learned features (edges, textures, patterns)

    include_top=False,
    # Remove the original classification layer of EfficientNet
    # A custom classifier will be added below

    input_shape=(IMG_SIZE, IMG_SIZE, 3)
    # Input image shape: (height, width, channels)
    # 3 = RGB images
)

In [ ]:
# Initially freeze the base model
# This keeps the pretrained weights unchanged during the first training phase
base_model.trainable = False

In [ ]:
# Custom Classifier Layers

x = base_model.output
# Output from EfficientNet (feature maps)

x = GlobalAveragePooling2D()(x)
# Converts 2D feature maps into a single vector
# Reduces dimensions while keeping important information

x = BatchNormalization()(x)
# Normalizes values to make training more stable and faster

x = Dense(512, activation='relu')(x)
# First dense layer to learn high-level patterns
# ReLU helps avoid vanishing gradients and speeds up learning

x = Dropout(0.4)(x)
# Randomly disables 40% of neurons during training
# Helps prevent overfitting

x = Dense(256, activation='relu')(x)
# Second dense layer to refine learned features

x = Dropout(0.3)(x)
# Additional regularization to improve generalization

output = Dense(NUM_CLASSES, activation='softmax')(x)
# Final output layer
# Softmax converts raw scores into probabilities for each class

In [ ]:
model = Model(inputs=base_model.input, outputs=output)

In [ ]:
model.summary()
# Displays the full layer structure and parameter count

---
## Model Compilation

Configures the model for training by specifying the optimizer, loss function, and evaluation metric.

| Setting | Value | Reason |
|---------|-------|--------|
| Optimizer | Adam (lr=1e-3) | Adaptive gradient descent, well-suited for image classification tasks |
| Loss | categorical_crossentropy | Standard loss for multi-class classification problems |
| Metric | accuracy | Measures the percentage of correct predictions per epoch |

In [ ]:
# Phase 1 compilation
# Base model is frozen, only the classifier head is trained
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    # Adam optimizer with a standard learning rate for the first phase

    loss='categorical_crossentropy',
    # Suitable for multi-class classification

    metrics=['accuracy']
    # Track prediction accuracy per epoch
)

print('Model compiled.')

---
## Model Fitting

The model is trained on the training dataset and its performance is monitored on the validation set after each epoch. The model updates its weights using gradient descent, and metrics like accuracy and loss are tracked to observe how well the model is learning.

| Callback | Purpose |
|----------|---------|
| EarlyStopping | Stops training if validation accuracy stops improving, preventing overfitting and wasted computation |
| ModelCheckpoint | Saves the best version of the model during training |
| ReduceLROnPlateau | Reduces the learning rate when progress stalls, allowing more precise weight updates |

In [ ]:
# Callbacks used in both Phase 1 and Phase 2
early_stop = EarlyStopping(
    monitor='val_accuracy',
    # Watch validation accuracy
    patience=5,
    # Stop if no improvement for 5 consecutive epochs
    restore_best_weights=True,
    # Restore the weights from the best epoch when stopped
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_model.keras',
    # File to save the best model
    monitor='val_accuracy',
    save_best_only=True,
    # Only overwrite when validation accuracy improves
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    # Halve the learning rate when triggered
    patience=3,
    # Trigger after 3 epochs with no improvement in validation loss
    verbose=1,
    min_lr=1e-7
    # Minimum allowed learning rate
)

# Phase 1: Train only the classifier head
history = model.fit(
    train_data,
    # Training images, model learns from these each epoch
    validation_data=test_data,
    # Testing images, used to monitor performance each epoch
    epochs=10,
    # Maximum number of full passes through the training dataset
    callbacks=[early_stop, checkpoint, reduce_lr]
)

print('Phase 1 training complete.')
print(f'Epochs run    : {len(history.history["accuracy"])}')
print(f'Best val acc  : {max(history.history["val_accuracy"]):.4f}')
print(f'Best val loss : {min(history.history["val_loss"]):.4f}')

---
## Fine-Tuning Setup

After the classifier head has been trained, the base model is partially unfrozen so that its deeper layers can also adapt to the skin disease dataset. Earlier layers are kept frozen because they contain general features such as edges and textures that remain useful. Only the last 30 layers are set to trainable.

The model is recompiled with a much lower learning rate to avoid overwriting the pretrained weights too aggressively.

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Freeze early layers, only train the last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False
# Earlier layers retain general features (edges, textures)
# Deeper layers adapt to the specific dataset

# Recompile with a lower learning rate
# Required after changing which layers are trainable
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    # Lower learning rate to avoid damaging pretrained weights

    loss='categorical_crossentropy',

    metrics=['accuracy']
)

trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
print('Fine-tuning setup complete.')
print(f'Trainable base layers : {trainable_count}')

In [ ]:
# Phase 2: Fine-tuning training
fine_tune_history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

print('Fine-tuning complete.')
print(f'Epochs run    : {len(fine_tune_history.history["accuracy"])}')
print(f'Best val acc  : {max(fine_tune_history.history["val_accuracy"]):.4f}')
print(f'Best val loss : {min(fine_tune_history.history["val_loss"]):.4f}')

---
## Training and Validation Metrics

Plots accuracy and loss across all epochs from both training phases.
- Accuracy should rise and converge for both training and validation
- Loss should decrease and converge
- If training accuracy improves but validation does not, the model is overfitting

In [ ]:
# Combine Phase 1 and Phase 2 history for a single continuous plot
combined_acc      = history.history['accuracy']     + fine_tune_history.history['accuracy']
combined_val_acc  = history.history['val_accuracy'] + fine_tune_history.history['val_accuracy']
combined_loss     = history.history['loss']         + fine_tune_history.history['loss']
combined_val_loss = history.history['val_loss']     + fine_tune_history.history['val_loss']
phase_split       = len(history.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy plot
ax1.plot(combined_acc,     label='Train Accuracy')
ax1.plot(combined_val_acc, label='Validation Accuracy')
ax1.axvline(phase_split - 1, color='gray', linestyle='dotted', label='Fine-tune start')
ax1.set_title('Accuracy per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(alpha=0.3)

# Loss plot
ax2.plot(combined_loss,     label='Train Loss')
ax2.plot(combined_val_loss, label='Validation Loss')
ax2.axvline(phase_split - 1, color='gray', linestyle='dotted', label='Fine-tune start')
ax2.set_title('Loss per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Training and Validation Metrics - EfficientNetB0', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Evaluation on Unseen Data

Evaluates the trained model on the unseen set, which contains images not encountered during training or testing. This provides the most reliable measure of real-world performance.

In [ ]:
# Overall accuracy and loss on unseen data
loss, acc = model.evaluate(unseen_data, verbose=1)
print('\n' + '=' * 45)
print(f'  Unseen Accuracy : {acc:.4f}  ({acc*100:.1f}%)')
print(f'  Unseen Loss     : {loss:.4f}')
print('=' * 45)

# Per-class report: precision, recall, and F1 score per disease
y_pred = np.argmax(model.predict(unseen_data), axis=1)
y_true = unseen_data.classes
class_names = list(unseen_data.class_indices.keys())

print('\nPer-class Report:\n')
print(classification_report(y_true, y_pred, target_names=class_names))

---
## Save and Download Model

In [ ]:
model.save('skin_disease_efficientnetb0.keras')
files.download('skin_disease_efficientnetb0.keras')
print('Model saved and downloaded.')